# Financial Fraud Detection System (Groq Version)

This notebook contains your fraud detection system code and the output shown below it.

**Security note:** the exposed Groq API key has been replaced with a placeholder in this notebook.


In [ ]:
from openai import OpenAI
import os
from typing import Dict, Any
from datetime import datetime
import re
import hashlib
import sys

# Initialize client (Groq compatible)
client = OpenAI(
    base_url="https://api.groq.com/openai/v1",
    api_key="YOUR_GROQ_API_KEY_HERE"
)

class FraudDetectionSystem:
    def __init__(self):
        self.suspicious_patterns = {
            "high_risk_countries": ["nigeria", "ghana", "russia", "north korea", "iran", "myanmar", "afghanistan"],
            "suspicious_keywords": ["overseas", "foreign", "international", "offshore", "wire transfer", "beneficiary"],
            "unusual_patterns": ["multiple deposits", "round amounts", "structuring", "just under", "split payment"],
            "velocity_indicators": ["fast transfer", "immediate", "urgent", "asap", "quick", "express"],
            "amount_thresholds": [10000, 50000, 100000]
        }
        self.available_models = {
            "versatile": "llama-3.3-70b-versatile",
            "instant": "llama-3.1-8b-instant",
            "vision": "meta-llama/llama-4-scout-17b-16e-instruct"
        }

    def query_ai(self, prompt: str, system_message: str = None, stream: bool = False) -> str:
        messages = []

        if system_message:
            messages.append({"role": "system", "content": system_message})

        messages.append({"role": "user", "content": prompt})

        try:
            completion = client.chat.completions.create(
                model=self.available_models["versatile"],
                messages=messages,
                temperature=0.7,
                max_tokens=2000,
                top_p=1,
                stream=stream
            )

            if stream:
                response = ""
                for chunk in completion:
                    if chunk.choices[0].delta.content:
                        content = chunk.choices[0].delta.content
                        print(content, end="", flush=True)
                        response += content
                print()
                return response
            else:
                return completion.choices[0].message.content

        except Exception as e:
            return f"API Error: {str(e)}"

    def analyze_transaction_deep(self, transaction: str) -> Dict[str, Any]:
        system_prompt = """You are a senior bank fraud detection expert with 20 years experience at major financial institutions. 
        Analyze transactions for:
        1. AML (Anti-Money Laundering) indicators
        2. Unusual patterns and red flags
        3. High-risk jurisdictions
        4. Structuring/smurfing attempts
        5. Behavioral anomalies
        6. Velocity checks
        7. Sanctions screening

        Be thorough, professional, and precise in your analysis."""

        analysis_prompt = f"""Analyze this transaction in detail and provide structured output:

        TRANSACTION DETAILS: {transaction}

        Please provide:
        1. Risk Assessment (Low/Medium/High/Critical)
        2. Risk Score (0-100)
        3. Specific Red Flags Found
        4. Recommended Actions
        5. Compliance Requirements
        6. Additional Verification Needed

        Format your response clearly with sections."""

        print("\n🤖 AI Analyzing Transaction...\n")
        analysis = self.query_ai(analysis_prompt, system_prompt, stream=True)

        return {
            "raw_analysis": analysis,
            "risk_score": self.calculate_risk_score(transaction),
            "timestamp": datetime.now().isoformat(),
            "transaction": transaction
        }

    def calculate_risk_score(self, transaction: str) -> int:
        score = 0
        transaction_lower = transaction.lower()

        print("\n🔍 Risk Factor Analysis:")
        print("-" * 40)

        if transaction.startswith('-f') or '/kernel-' in transaction:
            print("  ⚠️ Invalid transaction format detected")
            return 0

        for country in self.suspicious_patterns["high_risk_countries"]:
            if country in transaction_lower:
                score += 30
                print(f"  ⚠️ High-risk country detected: {country.title()}")

        for keyword in self.suspicious_patterns["suspicious_keywords"]:
            if keyword in transaction_lower:
                score += 20
                print(f"  🔍 Suspicious keyword: '{keyword}'")

        for pattern in self.suspicious_patterns["unusual_patterns"]:
            if pattern in transaction_lower:
                score += 15
                print(f"  📊 Unusual pattern: {pattern}")

        for indicator in self.suspicious_patterns["velocity_indicators"]:
            if indicator in transaction_lower:
                score += 10
                print(f"  ⚡ Velocity indicator: {indicator}")

        amounts = re.findall(r'\$?(\d+,?\d*\.?\d*)', transaction)
        for amount in amounts:
            try:
                amount_clean = float(amount.replace(',', ''))
                if amount_clean > 10000:
                    score += 25
                    print(f"  💰 Large amount detected: ${amount_clean:,.2f}")
                if amount_clean > 50000:
                    score += 35
                    print(f"  💰💰 Very large amount: ${amount_clean:,.2f}")
            except:
                pass

        if re.search(r'\d+\.?\d*\|?\d*', transaction):
            score += 20

        return min(score, 100)

    def generate_fraud_report(self, transaction: str, risk_score: int, analysis: str) -> str:
        report_prompt = f"""Generate a professional fraud detection report based on:

        TRANSACTION: {transaction}
        RISK SCORE: {risk_score}/100
        ANALYSIS: {analysis}

        Create a comprehensive compliance report including:
        1. Executive Summary
        2. Risk Assessment Matrix
        3. Identified Red Flags
        4. Regulatory Implications
        5. Required Actions
        6. SAR (Suspicious Activity Report) Filing Requirements
        7. Recommended Controls

        Format as a formal bank compliance document."""

        print("\n📝 Generating Compliance Report...\n")
        report = self.query_ai(report_prompt, stream=True)
        return report

    def verify_transaction(self, transaction: str, risk_score: int) -> str:
        if transaction.startswith('-f') or '/kernel-' in transaction:
            return "Invalid transaction format detected. Please provide a valid transaction description."

        verify_prompt = f"""Transaction approved with risk score {risk_score}/100.

        TRANSACTION: {transaction}

        Generate a confirmation message including:
        1. Approval confirmation with reference number
        2. Risk assessment summary
        3. Monitoring recommendations
        4. Next steps

        Keep it professional but clear."""

        print("\n✅ Processing Transaction...\n")
        verification = self.query_ai(verify_prompt, stream=True)
        return verification

    def generate_transaction_id(self, transaction: str) -> str:
        timestamp = datetime.now().strftime('%Y%m%d%H%M%S')
        hash_obj = hashlib.md5(transaction.encode())
        short_hash = hash_obj.hexdigest()[:6].upper()
        return f"TXN{timestamp}{short_hash}"

def fraud_detection_agent(transaction: str):
    if not transaction or transaction.startswith('-f') or '/kernel-' in transaction:
        transaction = "Sample transaction for testing: International wire transfer $5,000"
        print("⚠️ Using sample transaction for demonstration")

    print("=" * 80)
    print("🏦 FINANCIAL FRAUD DETECTION SYSTEM".center(80))
    print("=" * 80)

    detector = FraudDetectionSystem()
    transaction_id = detector.generate_transaction_id(transaction)

    print(f"\n📋 Transaction Review Summary")
    print(f"   ID: {transaction_id}")
    print(f"   Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}")
    print(f"   Type: {transaction[:50]}..." if len(transaction) > 50 else f"   Type: {transaction}")
    print("-" * 80)

    print("\n📊 RISK ASSESSMENT IN PROGRESS...")
    risk_score = detector.calculate_risk_score(transaction)

    print("\n" + "-" * 40)
    print(f"FINAL RISK SCORE: {risk_score}/100")

    if risk_score >= 75:
        risk_level = "CRITICAL RISK"
        risk_icon = "🔴"
    elif risk_score >= 50:
        risk_level = "HIGH RISK"
        risk_icon = "🟠"
    elif risk_score >= 25:
        risk_level = "MEDIUM RISK"
        risk_icon = "🟡"
    else:
        risk_level = "LOW RISK"
        risk_icon = "🟢"

    print(f"RISK LEVEL: {risk_icon} {risk_level}")
    print("-" * 40)

    print("\n🧠 ADVANCED TRANSACTION ANALYSIS:")
    analysis_result = detector.analyze_transaction_deep(transaction)

    print("\n" + "-" * 80)

    if risk_score >= 50 or any(country in transaction.lower() for country in ["nigeria", "ghana", "russia"]):
        print("\n🚨 FRAUD ALERT TRIGGERED")
        print("Generating Suspicious Activity Report...\n")

        final_output = detector.generate_fraud_report(
            transaction, 
            risk_score, 
            analysis_result["raw_analysis"]
        )

        alert_banner = """
╔════════════════════════════════════════════════════════════════╗
║                    🚨 FRAUD ALERT REPORT 🚨                     ║
╚════════════════════════════════════════════════════════════════╝
"""
        result_type = alert_banner
    else:
        print("\n✓ TRANSACTION VERIFIED")
        print("Processing normal transaction...\n")

        final_output = detector.verify_transaction(transaction, risk_score)

        confirm_banner = """
╔════════════════════════════════════════════════════════════════╗
║                 ✓ TRANSACTION CONFIRMATION ✓                   ║
╚════════════════════════════════════════════════════════════════╝
"""
        result_type = confirm_banner

    print("\n" + "=" * 80)
    print("FINAL REPORT".center(80))
    print("=" * 80)

    return f"""
Transaction ID: {transaction_id}
Date/Time: {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}
Risk Score: {risk_score}/100
Risk Classification: {risk_level}
Status: {'UNDER REVIEW' if risk_score >= 50 else 'APPROVED'}

{result_type}
{final_output}

{'=' * 80}
END OF REPORT
{'=' * 80}
"""

if __name__ == "__main__":
    if len(sys.argv) > 1:
        transaction = " ".join(sys.argv[1:])
    else:
        transaction = "Payment to overseas account in Nigeria for $15,000 - urgent transfer"

    print(fraud_detection_agent(transaction))


## Output

In [1]:
print(r"""⚠️ Using sample transaction for demonstration
================================================================================
                       🏦 FINANCIAL FRAUD DETECTION SYSTEM                       
================================================================================

📋 Transaction Review Summary
   ID: TXN202603180703551F8D4F
   Time: 2026-03-18 07:03:55
   Type: Sample transaction for testing: International wire...
--------------------------------------------------------------------------------

📊 RISK ASSESSMENT IN PROGRESS...

🔍 Risk Factor Analysis:
----------------------------------------
  🔍 Suspicious keyword: 'international'
  🔍 Suspicious keyword: 'wire transfer'

----------------------------------------
FINAL RISK SCORE: 60/100
RISK LEVEL: 🟠 HIGH RISK
----------------------------------------

🧠 ADVANCED TRANSACTION ANALYSIS:

🤖 AI Analyzing Transaction...

**Transaction Analysis Report**
================================

### Transaction Details
The transaction in question is an international wire transfer of $5,000.

### 1. Risk Assessment
Based on the provided information, the risk assessment for this transaction is: **Medium**

### 2. Risk Score
The risk score for this transaction is: **40**

### 3. Specific Red Flags Found
The following red flags were identified:
- **International Wire Transfer**: The transaction involves an international wire transfer, which can be a common method for money laundering or terrorist financing.
- **Lack of Transaction History**: There is no information provided about the transaction history between the sender and recipient, which makes it difficult to assess the legitimacy of the transaction.
- **No Information on Recipient**: The recipient's identity and location are not provided, which raises concerns about the potential for money laundering or terrorist financing.

### 4. Recommended Actions
The following actions are recommended:
- **Verify the Identity of the Sender and Recipient**: Confirm the identities of both parties involved in the transaction to ensure they are legitimate and not on any sanctions lists.
- **Analyze Transaction History**: Review the transaction history between the sender and recipient to identify any suspicious patterns or activities.
- **Monitor for Structuring**: Be aware of potential structuring attempts, where multiple transactions are made below the reporting threshold to avoid detection.

### 5. Compliance Requirements
The following compliance requirements must be met:
- **Know Your Customer (KYC)**: Ensure that the sender and recipient are subject to adequate KYC procedures to verify their identities and assess their risk profiles.
- **Anti-Money Laundering (AML) Regulations**: Comply with relevant AML regulations, including reporting suspicious transactions and maintaining accurate records.
- **Sanctions Screening**: Screen the sender and recipient against sanctions lists to ensure they are not subject to any restrictions.

### 6. Additional Verification Needed
The following additional verification is needed:
- **Purpose of the Transaction**: Clarify the purpose of the transaction to ensure it is legitimate and not related to any illicit activities.
- **Recipient's Country of Residence**: Verify the recipient's country of residence to assess the risk associated with the transaction, including the risk of money laundering or terrorist financing.
- **Sender's Source of Funds**: Confirm the sender's source of funds to ensure they are legitimate and not derived from any illicit activities.

🔍 Risk Factor Analysis:
----------------------------------------
  🔍 Suspicious keyword: 'international'
  🔍 Suspicious keyword: 'wire transfer'

--------------------------------------------------------------------------------

🚨 FRAUD ALERT TRIGGERED
Generating Suspicious Activity Report...


📝 Generating Compliance Report...

**Fraud Detection and Compliance Report**

**Transaction Reference:** International Wire Transfer of $5,000

**Date:** [Current Date]

**Report Number:** [Report Number]

---

### 1. Executive Summary

This report provides an analysis of a recent international wire transfer of $5,000, which has been flagged for potential fraud and non-compliance with Anti-Money Laundering (AML) and Know Your Customer (KYC) regulations. The transaction has a risk score of 60/100, indicating a medium to high risk of illicit activity. This report outlines the identified red flags, regulatory implications, required actions, and recommended controls to mitigate potential risks.

### 2. Risk Assessment Matrix

| **Risk Category** | **Risk Level** | **Description** |
| --- | --- | --- |
| Transaction Type | Medium | International wire transfer |
| Transaction Amount | Medium | $5,000 |
| Sender/Recipient Information | High | Lack of information on recipient's identity and location |
| Transaction History | Medium | No information provided on transaction history between sender and recipient |
| Geographical Risk | Medium | International transaction with potential for money laundering or terrorist financing |

### 3. Identified Red Flags

The following red flags have been identified in this transaction:

1. **International Wire Transfer**: The transaction involves an international wire transfer, which can be a common method for money laundering or terrorist financing.
2. **Lack of Transaction History**: There is no information provided about the transaction history between the sender and recipient, which makes it difficult to assess the legitimacy of the transaction.
3. **No Information on Recipient**: The recipient's identity and location are not provided, which raises concerns about the potential for money laundering or terrorist financing.

### 4. Regulatory Implications

This transaction may be subject to the following regulatory requirements:

1. **Anti-Money Laundering (AML) Regulations**: The transaction may be subject to AML regulations, including reporting suspicious transactions and maintaining accurate records.
2. **Know Your Customer (KYC) Regulations**: The sender and recipient must be subject to adequate KYC procedures to verify their identities and assess their risk profiles.
3. **Sanctions Screening**: The sender and recipient must be screened against sanctions lists to ensure they are not subject to any restrictions.

### 5. Required Actions

To mitigate potential risks and ensure compliance with regulatory requirements, the following actions are required:

1. **Verify the Identity of the Sender and Recipient**: Confirm the identities of both parties involved in the transaction to ensure they are legitimate and not on any sanctions lists.
2. **Analyze Transaction History**: Review the transaction history between the sender and recipient to identify any suspicious patterns or activities.
3. **Monitor for Structuring**: Be aware of potential structuring attempts, where multiple transactions are made below the reporting threshold to avoid detection.
4. **Clarify the Purpose of the Transaction**: Clarify the purpose of the transaction to ensure it is legitimate and not related to any illicit activities.
5. **Verify the Recipient's Country of Residence**: Verify the recipient's country of residence to assess the risk associated with the transaction, including the risk of money laundering or terrorist financing.
6. **Confirm the Sender's Source of Funds**: Confirm the sender's source of funds to ensure they are legitimate and not derived from any illicit activities.

### 6. SAR (Suspicious Activity Report) Filing Requirements

Based on the identified red flags and risk assessment, a SAR may be required to be filed with the relevant regulatory authority. The SAR should include the following information:

1. **Transaction Details**: The transaction amount, date, and type.
2. **Sender and Recipient Information**: The identities and locations of the sender and recipient.
3. **Red Flags**: The identified red flags, including the international wire transfer, lack of transaction history, and no information on recipient.
4. **Risk Assessment**: The risk assessment and scoring, including the medium to high risk of illicit activity.

### 7. Recommended Controls

To prevent similar transactions from being processed in the future, the following controls are recommended:

1. **Enhanced KYC Procedures**: Implement enhanced KYC procedures to verify the identities and risk profiles of senders and recipients.
2. **Transaction Monitoring**: Implement transaction monitoring systems to detect suspicious patterns and activities.
3. **Sanctions Screening**: Implement sanctions screening to ensure that senders and recipients are not subject to any restrictions.
4. **Employee Training**: Provide regular training to employees on AML and KYC regulations, as well as on identifying and reporting suspicious transactions.
5. **Audit and Compliance**: Regularly review and update policies and procedures to ensure compliance with regulatory requirements and to identify areas for improvement.

---

**Approved By:** [Name]
**Title:** [Title]
**Date:** [Date]

Note: This report is for illustration purposes only and should not be used in actual compliance reporting without proper modification and review by a qualified compliance professional.

================================================================================
                                  FINAL REPORT                                  
================================================================================

Transaction ID: TXN202603180703551F8D4F
Date/Time: 2026-03-18 07:03:59
Risk Score: 60/100
Risk Classification: HIGH RISK
Status: UNDER REVIEW


╔════════════════════════════════════════════════════════════════╗
║                    🚨 FRAUD ALERT REPORT 🚨                     ║
╚════════════════════════════════════════════════════════════════╝

**Fraud Detection and Compliance Report**

**Transaction Reference:** International Wire Transfer of $5,000

**Date:** [Current Date]

**Report Number:** [Report Number]

---

### 1. Executive Summary

This report provides an analysis of a recent international wire transfer of $5,000, which has been flagged for potential fraud and non-compliance with Anti-Money Laundering (AML) and Know Your Customer (KYC) regulations. The transaction has a risk score of 60/100, indicating a medium to high risk of illicit activity. This report outlines the identified red flags, regulatory implications, required actions, and recommended controls to mitigate potential risks.

### 2. Risk Assessment Matrix

| **Risk Category** | **Risk Level** | **Description** |
| --- | --- | --- |
| Transaction Type | Medium | International wire transfer |
| Transaction Amount | Medium | $5,000 |
| Sender/Recipient Information | High | Lack of information on recipient's identity and location |
| Transaction History | Medium | No information provided on transaction history between sender and recipient |
| Geographical Risk | Medium | International transaction with potential for money laundering or terrorist financing |

### 3. Identified Red Flags

The following red flags have been identified in this transaction:

1. **International Wire Transfer**: The transaction involves an international wire transfer, which can be a common method for money laundering or terrorist financing.
2. **Lack of Transaction History**: There is no information provided about the transaction history between the sender and recipient, which makes it difficult to assess the legitimacy of the transaction.
3. **No Information on Recipient**: The recipient's identity and location are not provided, which raises concerns about the potential for money laundering or terrorist financing.

### 4. Regulatory Implications

This transaction may be subject to the following regulatory requirements:

1. **Anti-Money Laundering (AML) Regulations**: The transaction may be subject to AML regulations, including reporting suspicious transactions and maintaining accurate records.
2. **Know Your Customer (KYC) Regulations**: The sender and recipient must be subject to adequate KYC procedures to verify their identities and assess their risk profiles.
3. **Sanctions Screening**: The sender and recipient must be screened against sanctions lists to ensure they are not subject to any restrictions.

### 5. Required Actions

To mitigate potential risks and ensure compliance with regulatory requirements, the following actions are required:

1. **Verify the Identity of the Sender and Recipient**: Confirm the identities of both parties involved in the transaction to ensure they are legitimate and not on any sanctions lists.
2. **Analyze Transaction History**: Review the transaction history between the sender and recipient to identify any suspicious patterns or activities.
3. **Monitor for Structuring**: Be aware of potential structuring attempts, where multiple transactions are made below the reporting threshold to avoid detection.
4. **Clarify the Purpose of the Transaction**: Clarify the purpose of the transaction to ensure it is legitimate and not related to any illicit activities.
5. **Verify the Recipient's Country of Residence**: Verify the recipient's country of residence to assess the risk associated with the transaction, including the risk of money laundering or terrorist financing.
6. **Confirm the Sender's Source of Funds**: Confirm the sender's source of funds to ensure they are legitimate and not derived from any illicit activities.

### 6. SAR (Suspicious Activity Report) Filing Requirements

Based on the identified red flags and risk assessment, a SAR may be required to be filed with the relevant regulatory authority. The SAR should include the following information:

1. **Transaction Details**: The transaction amount, date, and type.
2. **Sender and Recipient Information**: The identities and locations of the sender and recipient.
3. **Red Flags**: The identified red flags, including the international wire transfer, lack of transaction history, and no information on recipient.
4. **Risk Assessment**: The risk assessment and scoring, including the medium to high risk of illicit activity.

### 7. Recommended Controls

To prevent similar transactions from being processed in the future, the following controls are recommended:

1. **Enhanced KYC Procedures**: Implement enhanced KYC procedures to verify the identities and risk profiles of senders and recipients.
2. **Transaction Monitoring**: Implement transaction monitoring systems to detect suspicious patterns and activities.
3. **Sanctions Screening**: Implement sanctions screening to ensure that senders and recipients are not subject to any restrictions.
4. **Employee Training**: Provide regular training to employees on AML and KYC regulations, as well as on identifying and reporting suspicious transactions.
5. **Audit and Compliance**: Regularly review and update policies and procedures to ensure compliance with regulatory requirements and to identify areas for improvement.

---

**Approved By:** [Name]
**Title:** [Title]
**Date:** [Date]

Note: This report is for illustration purposes only and should not be used in actual compliance reporting without proper modification and review by a qualified compliance professional.

================================================================================
END OF REPORT
================================================================================""")

⚠️ Using sample transaction for demonstration
                       🏦 FINANCIAL FRAUD DETECTION SYSTEM                       

📋 Transaction Review Summary
   ID: TXN202603180703551F8D4F
   Time: 2026-03-18 07:03:55
   Type: Sample transaction for testing: International wire...
--------------------------------------------------------------------------------

📊 RISK ASSESSMENT IN PROGRESS...

🔍 Risk Factor Analysis:
----------------------------------------
  🔍 Suspicious keyword: 'international'
  🔍 Suspicious keyword: 'wire transfer'

----------------------------------------
FINAL RISK SCORE: 60/100
RISK LEVEL: 🟠 HIGH RISK
----------------------------------------

🧠 ADVANCED TRANSACTION ANALYSIS:

🤖 AI Analyzing Transaction...

**Transaction Analysis Report**

### Transaction Details
The transaction in question is an international wire transfer of $5,000.

### 1. Risk Assessment
Based on the provided information, the risk assessment for this transaction is: **Medium**

### 2. Risk Sco